# Baseline Model Training

This notebook builds baseline predictive models for the COMPAS responsible AI audit.

Please note: The goal is not to recommend the use of recidivism prediction systems. Instead, this notebook creates a simple model that can be audited in later notebooks for subgroup performance, false positive rates, false negative rates, and explainability.

In this notebook, race is not used as a model feature. However, demographic variables are retained separately for later fairness evaluation.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [ ]:
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"

df = pd.read_csv(url)

df.head()

,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


## Variable Selection


In [ ]:
model_df = df[
    [
        "age",
        "age_cat",
        "juv_fel_count",
        "juv_misd_count",
        "juv_other_count",
        "priors_count",
        "c_charge_degree",
        "sex",
        "race",
        "two_year_recid"
    ]
].copy()

model_df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,sex,race,two_year_recid
0,69,Greater than 45,0,0,0,0,F,Male,Other,0
1,34,25 - 45,0,0,0,0,F,Male,African-American,1
2,24,Less than 25,0,0,1,4,F,Male,African-American,1
3,23,Less than 25,0,1,0,1,F,Male,African-American,0
4,43,25 - 45,0,0,0,2,F,Male,Other,0


In [ ]:
model_df.isnull().sum()

,0
age,0
age_cat,0
juv_fel_count,0
juv_misd_count,0
juv_other_count,0
priors_count,0
c_charge_degree,0
sex,0
race,0
two_year_recid,0


### Interpretation: Modeling Variables and Missingness

The selected modeling dataset includes age-related variables, prior offense counts, juvenile offense counts, charge degree, and the target variable `two_year_recid`. Race and sex are retained in the dataset for later auditing, but they are not included as model features in this baseline model.

The missing-value check shows that the selected variables do not contain missing values. This makes the baseline modeling step more straightforward, but it does not remove the need for careful ethical interpretation.

**Please note:** In high-stakes criminal justice data, the absence of missing values does not mean the data are neutral, complete, or free from structural bias.


## Define Target/Features


In [ ]:
target = "two_year_recid"

# Race and sex are retained for auditing, but not included as model features here.
audit_columns = ["race", "sex"]

feature_columns = [
    "age",
    "age_cat",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree"
]

X = model_df[feature_columns]
y = model_df[target]

audit_data = model_df[audit_columns]

X.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree
0,69,Greater than 45,0,0,0,0,F
1,34,25 - 45,0,0,0,0,F
2,24,Less than 25,0,0,1,4,F
3,23,Less than 25,0,1,0,1,F
4,43,25 - 45,0,0,0,2,F


## Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test, audit_train, audit_test = train_test_split(
    X,
    y,
    audit_data,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 5410
Testing rows: 1804


### Interpretation: Train/Test Split

The data were split into training and testing sets using stratification on the target variable to help preserve the overall distribution of `two_year_recid` in both sets.

Race and sex were split alongside the features and target so they can be reattached to the test predictions later. This allows the project to evaluate subgroup error rates without using race or sex as predictive inputs.


## Pre-Processing Pipeline


In [ ]:
numeric_features = [
    "age",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count"
]

categorical_features = [
    "age_cat",
    "c_charge_degree"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

Numeric variables are standardized so they are placed on a comparable scale before logistic regression. Categorical variables are one-hot encoded so they can be used by the model.

## Logistic Regression Model


In [ ]:
log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

log_reg_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'juv_fel_count',
                                                   'juv_misd_count',
                                                   'juv_other_count',
                                                   'priors_count']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['age_cat',
                                                   'c_charge_degree'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

## Evaluation Metrics


In [ ]:
log_reg_preds = log_reg_model.predict(X_test)
log_reg_probs = log_reg_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, log_reg_preds))
print("Precision:", precision_score(y_test, log_reg_preds))
print("Recall:", recall_score(y_test, log_reg_preds))
print("F1 Score:", f1_score(y_test, log_reg_preds))
print("ROC-AUC:", roc_auc_score(y_test, log_reg_probs))

Accuracy: 0.6746119733924612
Precision: 0.6623563218390804
Recall: 0.5670356703567035
F1 Score: 0.6110006626905236
ROC-AUC: 0.7222846950972033


In [ ]:
print(classification_report(y_test, log_reg_preds))

              precision    recall  f1-score   support

           0       0.68      0.76      0.72       991
           1       0.66      0.57      0.61       813

    accuracy                           0.67      1804
   macro avg       0.67      0.66      0.67      1804
weighted avg       0.67      0.67      0.67      1804



In [ ]:
confusion_matrix(y_test, log_reg_preds)

array([[756, 235],
       [352, 461]])

### Logistic Regression Performance

The logistic regression model achieved approximately **67.5% accuracy**, with a **precision of 66.2%**, **recall of 56.7%**, **F1 score of 61.1%**, and **ROC-AUC of 72.2%**.

The confusion matrix shows **756 true negatives**, **235 false positives**, **352 false negatives**, and **461 true positives**. Meaning, the model correctly identifies many non-recidivism cases, but it also misses a substantial number of recidivism cases.

These aggregate results provide a useful baseline, but they do not show whether the false positives and false negatives are distributed evenly across demographic groups. That question will be addressed in the fairness evaluation notebook.


## Random Forest


In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'juv_fel_count',
                                                   'juv_misd_count',
                                                   'juv_other_count',
                                                   'priors_count']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['age_cat',
                                                   'c_charge_degree'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## Evaluation Metrics


In [ ]:
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds))
print("F1 Score:", f1_score(y_test, rf_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf_probs))

Accuracy: 0.6330376940133038
Precision: 0.5979247730220493
Recall: 0.5670356703567035
F1 Score: 0.5820707070707071
ROC-AUC: 0.6513970134655939


In [ ]:
print(classification_report(y_test, rf_preds))

              precision    recall  f1-score   support

           0       0.66      0.69      0.67       991
           1       0.60      0.57      0.58       813

    accuracy                           0.63      1804
   macro avg       0.63      0.63      0.63      1804
weighted avg       0.63      0.63      0.63      1804



In [ ]:
confusion_matrix(y_test, rf_preds)

array([[681, 310],
       [352, 461]])

### Random Forest Performance

The random forest model achieved approximately **63.3% accuracy**, with a **precision of 59.8%**, **recall of 56.7%**, **F1 score of 58.2%**, and **ROC-AUC of 65.1%**.

Compared with logistic regression, the random forest model has lower accuracy, lower precision, lower F1 score, and lower ROC-AUC in this baseline setup. Its recall is essentially the same as logistic regression, but it produces more false positives.

Because this project is focused on responsible AI auditing and explainability, logistic regression is a strong choice for the next steps. It performs better overall here and is also more interpretable than the random forest model.


## Model Comparison


In [ ]:
model_comparison = pd.DataFrame(
    {
        "Model": ["Logistic Regression", "Random Forest"],
        "Accuracy": [
            accuracy_score(y_test, log_reg_preds),
            accuracy_score(y_test, rf_preds)
        ],
        "Precision": [
            precision_score(y_test, log_reg_preds),
            precision_score(y_test, rf_preds)
        ],
        "Recall": [
            recall_score(y_test, log_reg_preds),
            recall_score(y_test, rf_preds)
        ],
        "F1 Score": [
            f1_score(y_test, log_reg_preds),
            f1_score(y_test, rf_preds)
        ],
        "ROC-AUC": [
            roc_auc_score(y_test, log_reg_probs),
            roc_auc_score(y_test, rf_probs)
        ]
    }
)

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.674612,0.662356,0.567036,0.611001,0.722285
1,Random Forest,0.633038,0.597925,0.567036,0.582071,0.651397


Logistic regression performs better than random forest on most aggregate metrics in this notebook. It has higher accuracy, precision, F1 score, and ROC-AUC, while recall is the same for both models.

For this reason, logistic regression will be used as the primary model in the fairness and explainability notebooks. This does not mean the model is ethically appropriate for deployment. It only means it is a reasonable baseline for demonstrating how responsible AI auditing can move beyond aggregate model performance.


In [ ]:
results_df = X_test.copy()
results_df["actual"] = y_test.values
results_df["log_reg_prediction"] = log_reg_preds
results_df["log_reg_probability"] = log_reg_probs
results_df["rf_prediction"] = rf_preds
results_df["rf_probability"] = rf_probs
results_df["race"] = audit_test["race"].values
results_df["sex"] = audit_test["sex"].values

results_df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,actual,log_reg_prediction,log_reg_probability,rf_prediction,rf_probability,race,sex
6421,30,25 - 45,0,0,0,2,F,1,0,0.427733,0,0.399627,African-American,Male
5898,55,Greater than 45,0,0,0,7,F,0,0,0.400000,0,0.255115,Caucasian,Male
3210,47,Greater than 45,0,0,0,2,F,0,0,0.310866,0,0.451463,Hispanic,Female
6651,25,25 - 45,0,0,0,2,F,0,0,0.488221,0,0.418322,African-American,Male
2811,54,Greater than 45,0,0,0,2,M,0,0,0.210459,0,0.436892,Hispanic,Female


### Preparing for Fairness Auditing

The `results_df` table combines the test-set features, actual outcomes, model predictions, prediction probabilities, and demographic audit columns. This structure is important because fairness auditing requires comparing model errors across groups.

This step also illustrates an important responsible AI principle: even when protected characteristics are not used as model inputs, they may still be necessary for auditing. Without subgroup information, it would be difficult to identify whether the model produces uneven error patterns.


## Modeling Summary

Two baseline models were trained: logistic regression and random forest.

The purpose of these models is to support later responsible AI auditing. Overall performance metrics such as accuracy, precision, recall, F1 score, and ROC-AUC provide a useful starting point, but they do not show whether the model performs differently across demographic groups.

The next notebook will examine subgroup performance and error rates, with particular attention to false positive and false negative rates.
